### Chargement du DataFrame

In [74]:
import pandas as pd
from pathlib import Path

ENTRAINEMENT = ""

if ENTRAINEMENT == "regex":
    df = pd.read_csv("../data/labelled_topics/dataset_annotated_regex.csv")
    MODEL_PATH = Path("../models/classification/logistic_regression/logistic_regression_regex.joblib")
    # Les données sont déjà équilibrées, on laisse le modèle faire son calcul naturellement
    poids_classes = None 
else:
    df = pd.read_csv("../data/labelled_topics/dataset_avis.csv")
    MODEL_PATH = Path("../models/classification/logistic_regression/logistic_regression.joblib")
    # Les classes sont déséquilibrées, on force le modèle à pénaliser les erreurs sur la classe minoritaire
    poids_classes = "balanced"

labels = ["qualité produit", "service livraison", "service client"]

df_annotated = df[df[labels].sum(axis=1) > 0]
df.shape

(2727, 16)

### Distribution des classes

In [75]:
class_distribution = df_annotated[labels].sum().to_frame(name="nb_commentaires")
class_distribution["pourcentage"] = (
    class_distribution["nb_commentaires"] / len(df_annotated) * 100
)

class_distribution

,nb_commentaires,pourcentage
qualité produit,1029,37.733773
service livraison,1417,51.961863
service client,447,16.391639


### Affichage d'avis au hasard

In [76]:
samples = []

pd.set_option("display.max_colwidth", None)

df_annotated = df_annotated.drop(columns=["client"], errors="ignore")

for label in labels:
    sample_df = df_annotated[df_annotated[label] == 1].sample(
        n=3,
        random_state=None
    )
    
    sample_df = sample_df.copy()
    sample_df["label_cible"] = label  # pour savoir pourquoi il est sélectionné
    
    samples.append(sample_df)

result = pd.concat(samples)

display_cols = result.rename(columns={
    "qualité produit": "produit",
    "service livraison": "livraison",
    "service client": "client"
})

display_cols = display_cols[[
    "label_cible",
    "produit",
    "livraison",
    "client",
    "clean_comment"
]]

display(display_cols)

# for _, row in display_cols.iterrows():
#     print("─" * 100)
#     print(f"Label cible : {row['label_cible']}")
#     print(f"Produit    : {row['produit']}")
#     print(f"Livraison  : {row['livraison']}")
#     print(f"Client    : {row['client']}")
#     print("\nCommentaire :")
#     print(row["clean_comment"])

,label_cible,produit,livraison,client,clean_comment
617,qualité produit,1,0,0,les photos sont bien prise et cela permet de valider la commande . et lors de la livraison on constate que le produit a bien été représenté lors des images sur le site .
274,qualité produit,1,1,0,je mets 1 étoile juste pour pouvoir ajouter mon avis plus que négatif ! 2eme livraison qui ne correspond en rien à ce que j ai commandé . le service client est lamentable me raccroche au nez après m avoir envoyée bouler ! ! ! ! je ne commande plus jamais sur ce site ! ! ! et je vire in extremis l application de mon smartphone . marre de se faire arnaquer ! !
344,qualité produit,1,0,0,"c ’ est vraiment des voleurs ! ! ! ! j ’ ai reçu un colis complètement cassé ( du verre ) je l ’ ai donc retourné .... on devait me rembourser 28€ et des poussières , or frais de livraison de 4€90 bien sur . et la surprise on me rembourse 8€ . ils m ’ ont volé 20€ . c ’ est la première fois que je commande chez showroom privé et bien ça sera la dernière fois ! ! ! ! de plus , impossible de les contacter pour leur réclamer le reste . vraiment des voleurs"
597,service livraison,0,1,0,livraison rapide et soignée
1251,service livraison,0,1,0,"je n'ai pas reçu la bonne couleur de casque . le service client m ' a juste demandé si je souhaitais le conserver , pas de dédommagement . je trouve qu'entre les colis endommagés car mal emballés , les articles d'une commande annulés au moment de l'envoi et les erreurs d'articles deviennent récurrents et le service client showroomprive ne dédommage pas ces erreurs par des gestes commerciaux ."
298,service livraison,0,1,0,"très déçu sur mon dernier achat ... je commande un beau sac en cuir jaune , je l attend pendant 2mois et au moment de la réception je me rends compte que le sac envoyé par showroom ne correspond pas du tt au model que j avais achète , je reçois un sac marron chocolat de très très mauvaise qualité et ne correspondant pas du tt a la photo du site.pas moyen de joindre showroom , retour par la poste , pas de point relai prêt de chez moi,8euros + 6,90 retiré sur mon remboursement.j ai payé 15euros pour renvoyer un sac , que je n ' avais même pas commandé.qu elle honte a l achat des frais de livraisonau retour des fraisau remboursement des fraiset tt ça pour rien ! ! ! ! je suis écoeuré.🤮fini pour moi showroom ."
608,service client,0,0,1,"qualité très soignée : matière , forme et finition . modèle seuant"
670,service client,0,1,1,rien à dire . livré dans les temps . produit conforme à la commande .
882,service client,0,0,1,"articles conformes à la description , livraison rapide et soignée . satisfaite de mes achats ."


### Création des variables X_test/y

In [77]:
X_text = df_annotated["clean_comment"].values
y = df_annotated[[
    "qualité produit",
    "service livraison",
    "service client"
]].values

### Encoding de X_text

In [78]:
from sentence_transformers import SentenceTransformer

model_emb = SentenceTransformer(
    "dangvantuan/french-document-embedding",
    trust_remote_code=True
)

X_embeddings = model_emb.encode(
    X_text,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

Batches:   0%|          | 0/86 [00:00<?, ?it/s]

### Séparation des données en train/test

In [79]:
from sklearn.model_selection import train_test_split
import numpy as np

indices = np.arange(len(df_annotated))

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_embeddings,
    y,
    indices,
    test_size=0.2,
    random_state=42
)

### Chargement ou création du modèle

In [80]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from joblib import dump, load

# Entraînement avec configuration dynamique
print("Configuration du modèle et recherche des hyperparamètres...")

log_reg = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    max_iter=2000,
    class_weight=poids_classes # None ou balanced selon les données d'entrainements (voir 1ere cellule)
)

# On cherche la meilleure régularisation pour éviter le surapprentissage
param_grid = {
    'estimator__C': [0.01, 0.1, 1.0, 10.0]
}

clf_base = OneVsRestClassifier(log_reg)
grid_search = GridSearchCV(clf_base, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)

grid_search.fit(X_train, y_train)

clf = grid_search.best_estimator_
print(f"Meilleur paramètre C trouvé : {grid_search.best_params_}")

# Sauvegarde
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
dump(clf, MODEL_PATH)

print(f"Modèle entraîné (Mode: {ENTRAINEMENT}) et sauvegardé sous : {MODEL_PATH}")

Configuration du modèle et recherche des hyperparamètres...
Meilleur paramètre C trouvé : {'estimator__C': 10.0}
Modèle entraîné (Mode: ) et sauvegardé sous : ..\models\classification\logistic_regression\logistic_regression.joblib


c:\IA\nov24_alt_trustpilot\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\IA\nov24_alt_trustpilot\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\IA\nov24_alt_trustpilot\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warn

### CrossVal

In [81]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, f1_score

scorer = make_scorer(f1_score, average="micro")

cv_scores = cross_val_score(
    clf,
    X_embeddings,
    y,
    cv=5,
    scoring=scorer,
    n_jobs=-1
)

print("F1 micro par fold :", cv_scores)
print("F1 micro moyen   :", cv_scores.mean())
print("Écart-type       :", cv_scores.std())

F1 micro par fold : [0.60223642 0.546851   0.56235107 0.52332016 0.59826361]
F1 micro moyen   : 0.5666044530819392
Écart-type       : 0.030178668366735314


### Calcul des probas sur test ou données labelisées

In [85]:
EVAL_MODE = "test"

if EVAL_MODE == "test":
    print("Évaluation sur le jeu de test")

    X_eval = X_test
    y_eval = y_test
elif EVAL_MODE == "labellise":
    print("Évaluation sur les données labelisées")

    df_eval = pd.read_csv("../data/test_dataset/100_avis_annote.csv", sep=";")

    X_eval_text = df_eval["clean_comment"].values
    y_eval = df_eval[[
        "qualité produit",
        "service livraison",
        "service client"
    ]].values
    X_eval = model_emb.encode(
        X_eval_text,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

y_proba = clf.predict_proba(X_eval)

Évaluation sur le jeu de test


### Affichage des résultats

In [86]:
from sklearn.metrics import classification_report

threshold = 0.5

y_pred = (y_proba >= threshold).astype(int)

print(classification_report(
    y_eval,
    y_pred,
    target_names=[
        "qualité produit",
        "service livraison",
        "service client"
    ]
))

                   precision    recall  f1-score   support

  qualité produit       0.60      0.79      0.69       226
service livraison       0.62      0.53      0.57       284
   service client       0.29      0.57      0.38        75

        micro avg       0.54      0.64      0.58       585
        macro avg       0.50      0.63      0.55       585
     weighted avg       0.57      0.64      0.59       585
      samples avg       0.54      0.64      0.57       585



c:\IA\nov24_alt_trustpilot\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [73]:
import numpy as np

labels = ["qualité produit", "service livraison", "service client"]

threshold = 0.50

thresholds = {
    "qualité produit": threshold,
    "service livraison": threshold,
    "service client": threshold
}

y_pred = np.zeros_like(y_proba, dtype=int)

for i, label in enumerate(labels):
    y_pred[:, i] = (y_proba[:, i] >= thresholds[label]).astype(int)

from sklearn.metrics import confusion_matrix

for i, label in enumerate(labels):
    tn, fp, fn, tp = confusion_matrix(
        y_eval[:, i],
        y_pred[:, i]
    ).ravel()
    
    df_cm = pd.DataFrame(
        [[tn, fp], [fn, tp]],
        index=["Vrai 0", "Vrai 1"],
        columns=["Prédit 0", "Prédit 1"]
    )
    
    print(f"\n{label}")
    display(df_cm)




qualité produit


,Prédit 0,Prédit 1
Vrai 0,32,27
Vrai 1,14,27



service livraison


,Prédit 0,Prédit 1
Vrai 0,48,31
Vrai 1,8,13



service client


,Prédit 0,Prédit 1
Vrai 0,71,2
Vrai 1,15,12


In [52]:
if EVAL_MODE == "test":
    df_eval = df_annotated.iloc[idx_test].copy()
elif EVAL_MODE == "labeled":
    df_eval = df_eval.copy()
pd.set_option("display.max_colwidth", None)
for i, label in enumerate(labels):
    df_eval[f"y_true_{label}"] = y_eval[:, i]
    df_eval[f"y_pred_{label}"] = y_pred[:, i]
    df_eval[f"proba_{label}"] = y_proba[:, i]

# Affichage de quelques avis (faux négatifs ou faux positifs)
def show_errors(df, label, n=10):
    y_true = f"y_true_{label}"
    y_pred = f"y_pred_{label}"

    proba_cols = [f"proba_{l}" for l in labels]

    fn = df[(df[y_true] == 1) & (df[y_pred] == 0)]
    fp = df[(df[y_true] == 0) & (df[y_pred] == 1)]

    print(f"\n{label.upper()} — Faux négatifs")
    display(fn[["clean_comment"] + proba_cols].head(n))

    print(f"\n{label.upper()} — Faux positifs")
    display(fp[["clean_comment"] + proba_cols].head(n))

for label in labels:
    show_errors(df_eval, label, n=10)


QUALITÉ PRODUIT — Faux négatifs


,clean_comment,proba_qualité produit,proba_service livraison,proba_service client
6,un peu juste en taille alors que j ai commandé taille l je fais du 42,0.438808,0.024813,0.199445
8,les chaussures reçues ne sont pas de la marque commandée ! papillio au lieu de birkenstock ! et sans être prévenue !,0.366467,0.060853,0.098599
11,bonjour j'ai reçu deux articles qui correspondent pas à ma commande la robe rouge évasée j'ai reçu carrément un autre article de la marque etam et la blouse le coloris n'est pas du tout la même que celle de l'image bien à vous,0.290669,0.064379,0.342383
15,"commande n°205309555 d'un sac laura ashley , le sac comporte un gros coup de cuter dans le bas . commande n°205309554 des braseros , il manque les vis de montage . très déçu du sérieux de ce pure player .",0.454165,0.043578,0.098875
17,par contre semelles trop plate,0.450325,0.005899,0.214721
18,pas encore essayé mais ces boucles ont l'air très bien,0.226442,0.014485,0.079217
28,vendu pour être du cuir mais ce n'en est clairement pas !,0.080071,0.143288,0.170654
37,au top . mais veste xl hyper serré !,0.489012,0.094563,0.244285
39,"bonjour , pour l'anniversaire de mon père j'ai décidé de commander des bières sur vente privé , mauvaise surprise en arrivant ( avec plus d'un mois d'attente sois disant passant ) 3 sur 6 bières sont vides ... aucune vérification n'est faite au moment de l'envoi , par conséquent je ne souhaite plus acheter chez eux . pour me dédommager ils m'ont proposé un `` bon d'achat '' 5€ ( montant des frais de port ) cela m'oblige donc à recommander chez eux . c'est donc pour moi de la achat forcée ! prenez garde avant de commander .",0.487656,0.163281,0.432770
43,"j'ai commandé 2 paires de chaussures pointure 40 et j'ai reçu une paire commandée en pointure 40 et une autre paire pointure 37 qui rien à voir avec ce que j'avais commandé . donc obligée de faire un retour pour mauvais article , perte de temps et déçue de ne pas avoir réceptionné l'autre paire de chaussure",0.276379,0.074771,0.368056



QUALITÉ PRODUIT — Faux positifs


,clean_comment,proba_qualité produit,proba_service livraison,proba_service client
1,bonjour je n ai toujours pas reçu ma commande,0.708230,0.025960,0.055152
7,"après réflexion j ’ ai souhaité modifier ma commande et ajouter un article résultat impossible du coup frais de port payé deux fois pour la même vente ... dommage , ni économique , ni écologique .",0.627768,0.048347,0.055838
10,boîtes de chaussures défoncées mais chaussures en très bon état quand même,0.665778,0.031566,0.117802
20,tres satisfaite rien à signaler,0.536500,0.034346,0.050891
23,"hormis les 2 mois d'attente , les articles correspondent à ma commande",0.647433,0.041604,0.157350
25,merci beaucoup pour votre rapidité,0.748205,0.137223,0.040963
31,je n'ai pas reçu l l'article commandé j'ai reçu un chemisier à carreaux marron et noir à la place d un chemisier blanc,0.516646,0.023165,0.243488
32,ma commande n ’ a pas pu être complète et je n ’ ai toujours pas était remboursé,0.571882,0.031709,0.078404
35,collection site irl non visiter à cause de showroom sur facebook,0.604944,0.045461,0.135123
40,satisfe de mon achat .,0.863567,0.044253,0.142998



SERVICE LIVRAISON — Faux négatifs


...

,clean_comment,proba_qualité produit,proba_service livraison,proba_service client



SERVICE CLIENT — Faux négatifs


,clean_comment,proba_qualité produit,proba_service livraison,proba_service client
7,"après réflexion j ’ ai souhaité modifier ma commande et ajouter un article résultat impossible du coup frais de port payé deux fois pour la même vente ... dommage , ni économique , ni écologique .",0.627768,0.048347,0.055838
8,les chaussures reçues ne sont pas de la marque commandée ! papillio au lieu de birkenstock ! et sans être prévenue !,0.366467,0.060853,0.098599
11,bonjour j'ai reçu deux articles qui correspondent pas à ma commande la robe rouge évasée j'ai reçu carrément un autre article de la marque etam et la blouse le coloris n'est pas du tout la même que celle de l'image bien à vous,0.290669,0.064379,0.342383
12,bien que j'ai été prévenue il manquait un élément à ma commande et c'est malheureusement pour le produit en question que je passais commande .,0.316283,0.061288,0.390383
13,bonjourj'ai commandé un tableau la commande est toujours en préparation depuis 15 jours site à éviter 😡,0.481317,0.015298,0.309478
15,"commande n°205309555 d'un sac laura ashley , le sac comporte un gros coup de cuter dans le bas . commande n°205309554 des braseros , il manque les vis de montage . très déçu du sérieux de ce pure player .",0.454165,0.043578,0.098875
21,dommage de ne pas avoir reçu le bon article . showroom est plutôt bien réputé pour faire des erreurs comme ça . joyeux noël charline,0.395354,0.037317,0.172743
22,pas terrible ! ! ! j étais bonne cliente et maintenant je commande très peu et il trouve dle moyen de se tromper d article ! ! ! ! au revoir showroom ..........,0.232328,0.055858,0.273475
23,"hormis les 2 mois d'attente , les articles correspondent à ma commande",0.647433,0.041604,0.157350
31,je n'ai pas reçu l l'article commandé j'ai reçu un chemisier à carreaux marron et noir à la place d un chemisier blanc,0.516646,0.023165,0.243488



SERVICE CLIENT — Faux positifs


,clean_comment,proba_qualité produit,proba_service livraison,proba_service client
97,je ne l'ai pas reçu je suis déçu car c'est un cadeau pour l'anniversaire de mon mari .,0.29785,0.101353,0.53003
